In [ ]:
import os

# Yahan apna dataset ka path daalo (Drive mein jahan folder rakha hai)
dataset_path = '/content/drive/MyDrive/Colab_Notebooks/Deep_Learning/CNN/Solar Panel Defect Classification/Data'

# Har class folder ka naam list karo
class_names = os.listdir(dataset_path)
print("Classes found:", class_names)

# Har class mein kitni images hain, ye count kar ke verify karte hain
# (taake pata chale hamara upar wala data count sahi match kar raha hai ya nahi)
for class_name in class_names:
    class_folder_path = os.path.join(dataset_path, class_name)

    # Sirf folders ko count karo, files ko skip karo
    if os.path.isdir(class_folder_path):
        image_files = os.listdir(class_folder_path)
        num_images = len(image_files)
        print(f"{class_name}: {num_images} images")

Classes found: ['Bird-drop', 'Clean', 'Dusty', 'Electrical-damage', 'Snow-Covered', 'Physical-Damage']
Bird-drop: 193 images
Clean: 194 images
Dusty: 190 images
Electrical-damage: 103 images
Snow-Covered: 123 images
Physical-Damage: 69 images


In [5]:
import os
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

In [6]:
!pip install split-folders

In [7]:
import splitfolders

# Dataset ko train(70%)/val(15%)/test(15%) mein split karo
# Ye original dataset_path ko nahi chhedta, naya folder banata hai
splitfolders.ratio(
    dataset_path,
    output='/content/split_dataset',
    seed=42,
    ratio=(0.7, 0.15, 0.15)
)

Copying files: 871 files [00:34, 25.40 files/s]


In [8]:
import tensorflow as tf

# Paths teeno split folders ke
train_path = '/content/split_dataset/train'
val_path = '/content/split_dataset/val'
test_path = '/content/split_dataset/test'

# Training dataset load karo
train_dataset = tf.keras.utils.image_dataset_from_directory(
    train_path,
    image_size=(224, 224),
    batch_size=32,
    seed=123
)

# Validation dataset load karo
val_dataset = tf.keras.utils.image_dataset_from_directory(
    val_path,
    image_size=(224, 224),
    batch_size=32,
    seed=123
)

# Test dataset load karo (final unseen evaluation ke liye)
test_dataset = tf.keras.utils.image_dataset_from_directory(
    test_path,
    image_size=(224, 224),
    batch_size=32,
    seed=123
)

# Class names verify karo (folder names se auto-detect hote hain)
class_names = train_dataset.class_names
print("Classes:", class_names)

Found 607 files belonging to 6 classes.
Found 127 files belonging to 6 classes.
Found 135 files belonging to 6 classes.
Classes: ['Bird-drop', 'Clean', 'Dusty', 'Electrical-damage', 'Physical-Damage', 'Snow-Covered']


In [9]:
normalization_layer = tf.keras.layers.Rescaling(1./255)

# Har dataset pe normalization apply karo
train_dataset = train_dataset.map(lambda x, y: (normalization_layer(x), y))
val_dataset = val_dataset.map(lambda x, y: (normalization_layer(x), y))
test_dataset = test_dataset.map(lambda x, y: (normalization_layer(x), y))

In [10]:
AUTOTUNE = tf.data.AUTOTUNE

# cache: pehli epoch ke baad images memory mein rakh leta hai (disk se dobara read nahi karna padta)
# shuffle: sirf training data pe - har epoch mein data order badalte rehna chahiye
# prefetch: agli batch ready rakhta hai jab current batch pe GPU train kar raha ho
train_dataset = train_dataset.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
val_dataset = val_dataset.cache().prefetch(buffer_size=AUTOTUNE)
test_dataset = test_dataset.cache().prefetch(buffer_size=AUTOTUNE)

In [11]:
from tensorflow.keras import layers, models

# Augmentation layers - ye sirf training ke time active hote hain
data_augmentation = models.Sequential([
    layers.RandomFlip("horizontal"),           # left-right flip
    layers.RandomRotation(0.1),                # thoda rotate (10% of 360 degrees)
    layers.RandomZoom(0.1),                    # thoda zoom in/out
    layers.RandomContrast(0.1),                # brightness/contrast variation
])

In [12]:
from tensorflow.keras import layers, models

# Pehle augmentation block ko alag se define karo


# Ab model banao, aur data_augmentation ko sirf naam se use karo (variable ke roop mein)
model = models.Sequential([

    # Input shape define karo
    layers.Input(shape=(224, 224, 3)),

    # Augmentation - yahan sirf variable ka naam daalna hai, define nahi karna
    data_augmentation,

    # Conv Block 1
    layers.Conv2D(32, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),

    # Conv Block 2
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),

    # Conv Block 3
    layers.Conv2D(128, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),

    # Flatten karo
    layers.Flatten(),

    # Dense layer
    layers.Dense(128, activation='relu'),

    # Dropout
    layers.Dropout(0.5),

    # Output layer
    layers.Dense(6, activation='softmax')
])

model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ sequential (Sequential)         │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 222, 222, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 109, 109, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 52, 52, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 26, 26, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 86528)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │    11,075,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 6)              │           774 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 11,169,734 (42.61 MB)

 Trainable params: 11,169,734 (42.61 MB)

 Non-trainable params: 0 (0.00 B)

In [13]:
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',  # kyunki labels int form mein hain (label_mode default)
    metrics=['accuracy']
)

In [14]:
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

# val_loss monitor karega - agar 7 epochs tak improve na ho to training rok dega
# restore_best_weights=True - training khatam hone pe best epoch wale weights wapas laga dega
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=7,
    restore_best_weights=True
)

# Best model ko file mein save karega (jab bhi val_loss improve ho)
checkpoint = ModelCheckpoint(
    'best_model.keras',
    monitor='val_loss',
    save_best_only=True
)


In [15]:
history = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=50,  # upar limit rakh do, EarlyStopping khud rok dega
    callbacks=[early_stop, checkpoint]
)

Epoch 1/50
19/19 ━━━━━━━━━━━━━━━━━━━━ 19s 206ms/step - accuracy: 0.2438 - loss: 2.1829 - val_accuracy: 0.3465 - val_loss: 1.5806
Epoch 2/50
19/19 ━━━━━━━━━━━━━━━━━━━━ 2s 92ms/step - accuracy: 0.3690 - loss: 1.6104 - val_accuracy: 0.3543 - val_loss: 1.5269
Epoch 3/50
19/19 ━━━━━━━━━━━━━━━━━━━━ 1s 54ms/step - accuracy: 0.3196 - loss: 1.6049 - val_accuracy: 0.3386 - val_loss: 1.5838
Epoch 4/50
19/19 ━━━━━━━━━━━━━━━━━━━━ 2s 93ms/step - accuracy: 0.4135 - loss: 1.4907 - val_accuracy: 0.4252 - val_loss: 1.4142
Epoch 5/50
19/19 ━━━━━━━━━━━━━━━━━━━━ 1s 58ms/step - accuracy: 0.4267 - loss: 1.4234 - val_accuracy: 0.3780 - val_loss: 1.5331
Epoch 6/50
19/19 ━━━━━━━━━━━━━━━━━━━━ 1s 63ms/step - accuracy: 0.4794 - loss: 1.4048 - val_accuracy: 0.4724 - val_loss: 1.4223
Epoch 7/50
19/19 ━━━━━━━━━━━━━━━━━━━━ 2s 102ms/step - accuracy: 0.4860 - loss: 1.3595 - val_accuracy: 0.5197 - val_loss: 1.2642
Epoch 8/50
19/19 ━━━━━━━━━━━━━━━━━━━━ 1s 56ms/step - accuracy: 0.4959 - loss: 1.3473 - val_accuracy: 0.4331 

In [16]:
best_model = tf.keras.models.load_model('best_model.keras')
loss, accuracy = best_model.evaluate(val_dataset)
print(f"Loaded best model - Val Loss: {loss:.4f}, Val Accuracy: {accuracy:.4f}")

4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.5827 - loss: 1.0838 
Loaded best model - Val Loss: 1.0838, Val Accuracy: 0.5827
